In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import matplotlib.ticker as mticker

In [ ]:
def make_lin_fit(xdata, ydata, log=True):
    if log:
        xdata = np.log(xdata)
        ydata = np.log(ydata)
    Clog = np.cov(np.vstack((xdata, ydata)))
    corrlog = Clog[0, 1] / np.sqrt(Clog[0, 0] * Clog[1, 1])
    eigVals, eigVecs = np.linalg.eig(Clog)
    max_eigval = np.argmax(eigVals)
    slope = eigVecs[1, max_eigval] / eigVecs[0, max_eigval]
    x_fit = np.linspace(xdata.min(), xdata.max(), 20)
    y_fit = slope * (x_fit - xdata.mean()) + ydata.mean()
    intercept = slope * (0 - xdata.mean()) + ydata.mean()
    return x_fit, y_fit, slope, intercept

In [ ]:
scaling_df = pd.read_csv('timescaling_normal.csv')
figure_folder = 'figures'

# fig_total, ax_total = plt.subplots()

# Plot timescaling n_cells vs time, but normalized to n_genes
n_cells = scaling_df['n_cells'].values
ordering = np.flatnonzero(~np.isnan(n_cells))[np.argsort(n_cells[~np.isnan(n_cells)])]
n_cells = n_cells[ordering]

n_secs_total = scaling_df['total_time'].values
n_secs_total = n_secs_total[ordering]
n_genes = scaling_df['n_genes'].values
n_genes = n_genes[ordering]
n_secs_total_normalized = n_secs_total / n_genes * 2000
max_memories_zero = scaling_df['peak_mem_zero'] * (1024 ** 2) / 1e9  # Correct for that it was reported in MiB, not MB
max_memories_zero = max_memories_zero[ordering]
max_memories_nonzero = scaling_df['peak_mem_avg_nonzero'] * (1024 ** 2) / 1e9
max_memories_nonzero = max_memories_nonzero[ordering]

# Make a plot for time-scaling
skip_first_n = 2
fig_time, ax_time = plt.subplots()
x_fit, y_fit, slope, intercept = make_lin_fit(n_cells[skip_first_n:], n_secs_total_normalized[skip_first_n:], log=True)
ax_time.plot(n_cells, n_secs_total_normalized, 'o', c='gray', lw=2, markersize=10, label='Default Bonsai')
ax_time.plot(np.exp(x_fit), np.exp(y_fit), '--', lw=2, c='black')

ax_time.set_xscale('log')
ax_time.set_yscale('log')
ax_time.legend()
ax_time.grid(True)
ax_time.grid(True, which='minor', lw=0.25, linestyle='--')
ax_time.set_xlabel("Number of cells")
ax_time.set_ylabel("Compute time (seconds) \n normalized per 2000 genes.")
ax_time.set_title(r'Fitted scaling: $T = {%.4f} |C|^{%.2f}$' % (np.exp(intercept), slope))
plt.savefig(os.path.join(figure_folder, 'core_compute_scaling.png'), dpi=300)
plt.savefig(os.path.join(figure_folder, 'core_compute_scaling.svg'))

# Make such a plot for memory scaling as well
skip_first_n = 2
fig_mem, ax_mem = plt.subplots()
x_fit, y_fit, slope, intercept = make_lin_fit(n_cells[skip_first_n:], max_memories_zero[skip_first_n:], log=True)
ax_mem.plot(n_cells, max_memories_zero, 'o', c='gray', lw=2, markersize=10, label=r'Main thread (Default Bonsai): $M = {%.4f} |C|^{%.2f}$' % (np.exp(intercept), slope))
ax_mem.plot(np.exp(x_fit), np.exp(y_fit), '--', lw=2, c='gray')

x_fit, y_fit, slope, intercept = make_lin_fit(n_cells[skip_first_n:], max_memories_nonzero[skip_first_n:], log=True)
ax_mem.plot(n_cells, max_memories_nonzero, 'o', c='blue', lw=2, markersize=10, label=r'Worker threads (Default Bonsai): $M = {%.4f} |C|^{%.2f}$' % (np.exp(intercept), slope))
ax_mem.plot(np.exp(x_fit), np.exp(y_fit), '--', lw=2, c='blue')

ax_mem.set_xscale('log')
ax_mem.set_yscale('log')
ax_mem.legend()
ax_mem.grid(True)
ax_mem.grid(True, which='minor', lw=0.25, linestyle='--')
ax_mem.set_xlabel("Number of cells")
ax_mem.set_ylabel("Memory usage (GB).")
# ax_mem.set_title(r'Fitted scaling: $T = {%.4f} |C|^{%.2f}$' % (np.exp(intercept), slope))
plt.savefig(os.path.join(figure_folder, 'memory_scaling.png'), dpi=300)
plt.savefig(os.path.join(figure_folder, 'memory_scaling.svg'))

## Make such a plot for iterative Bonsai as well

In [ ]:
scaling_df = pd.read_csv('timescaling_iterative.csv')
figure_folder = 'figures'

# fig_total, ax_total = plt.subplots()

# Plot timescaling n_cells vs time, but normalized to n_genes
n_cells = scaling_df['n_cells'].values
ordering = np.flatnonzero(~np.isnan(n_cells))[np.argsort(n_cells[~np.isnan(n_cells)])]
n_cells = n_cells[ordering]
# date_format = "%B %d, %H:%M:%S"
# n_secs_total = []
# for row_ind in range(scaling_df.shape[0]):
#     final_datetime = datetime.strptime(scaling_df['end_time_metadata'].iloc[row_ind], date_format)
#     first_datetime = datetime.strptime(scaling_df['start_time_preprocess'].iloc[row_ind], date_format)
#     n_secs_total.append((final_datetime - first_datetime).total_seconds())
n_secs_total = scaling_df['total_time'].values
n_secs_total = n_secs_total[ordering]
n_genes = scaling_df['n_genes'].values
n_genes = n_genes[ordering]
n_secs_total_normalized = n_secs_total / n_genes * 2000
max_memories_zero = scaling_df['peak_mem_zero']  * (1024 ** 2) / 1e9
max_memories_zero = max_memories_zero[ordering]
max_memories_nonzero = scaling_df['peak_mem_avg_nonzero'] * (1024 ** 2) / 1e9
max_memories_nonzero = max_memories_nonzero[ordering]

# Make a plot for time-scaling
skip_first_n = 2
fig_time, ax_time = plt.subplots()
x_fit, y_fit, slope, intercept = make_lin_fit(n_cells[skip_first_n:], n_secs_total_normalized[skip_first_n:], log=True)
ax_time.plot(n_cells, n_secs_total_normalized, 'o', c='gray', lw=2, markersize=10, label='Backbone-based Bonsai')
ax_time.plot(np.exp(x_fit), np.exp(y_fit), '--', lw=2, c='black')

ax_time.set_xscale('log')
ax_time.set_yscale('log')
ax_time.legend()
ax_time.grid(True)
ax_time.grid(True, which='minor', lw=0.25, linestyle='--')
ax_time.set_xlabel("Number of cells")
ax_time.set_ylabel("Compute time (seconds) \n normalized per 2000 genes.")
ax_time.set_title(r'Fitted scaling: $T = {%.4f} |C|^{%.2f}$' % (np.exp(intercept), slope))
plt.savefig(os.path.join(figure_folder, 'iterative_compute_scaling.png'), dpi=300)
plt.savefig(os.path.join(figure_folder, 'iterative_compute_scaling.svg'))

# Make such a plot for memory scaling as well
skip_first_n = 2
fig_mem, ax_mem = plt.subplots()
x_fit, y_fit, slope, intercept = make_lin_fit(n_cells[skip_first_n:], max_memories_zero[skip_first_n:], log=True)
ax_mem.plot(n_cells, max_memories_zero, 'o', c='gray', lw=2, markersize=10, label=r'Main thread (Backbone-based Bonsai): $M = {%.4f} |C|^{%.2f}$' % (np.exp(intercept), slope))
ax_mem.plot(np.exp(x_fit), np.exp(y_fit), '--', lw=2, c='gray')

x_fit, y_fit, slope, intercept = make_lin_fit(n_cells[skip_first_n:], max_memories_nonzero[skip_first_n:], log=True)
ax_mem.plot(n_cells, max_memories_nonzero, 'o', c='blue', lw=2, markersize=10, label=r'Worker threads (Backbone-based Bonsai): $M = {%.4f} |C|^{%.2f}$' % (np.exp(intercept), slope))
ax_mem.plot(np.exp(x_fit), np.exp(y_fit), '--', lw=2, c='blue')

ax_mem.set_xscale('log')
ax_mem.set_yscale('log')
ax_mem.legend()
ax_mem.grid(True)
ax_mem.grid(True, which='minor', lw=0.25, linestyle='--')
ax_mem.set_xlabel("Number of cells")
ax_mem.set_ylabel("Memory usage (GB).")
# ax_mem.set_title(r'Fitted scaling: $T = {%.4f} |C|^{%.2f}$' % (np.exp(intercept), slope))
plt.savefig(os.path.join(figure_folder, 'iterative_scaling.png'), dpi=300)
plt.savefig(os.path.join(figure_folder, 'iterative_scaling.svg'))

In [ ]:
steplabel_map = {0: 'Preprocessing', 1: 'Guide tree (1k cells)', 2: 'Add remaining cells', 3: 'Refine final'}

scaling_df = pd.read_csv('timescaling_iterative.csv')
figure_folder = 'figures'

colors = [plt.get_cmap("tab10")(i) for i in range(10)]

# fig_total, ax_total = plt.subplots()

# Plot timescaling n_cells vs time, but normalized to n_genes
n_cells = scaling_df['n_cells'].values
ordering = np.flatnonzero(~np.isnan(n_cells))[np.argsort(n_cells[~np.isnan(n_cells)])]
n_cells = n_cells[ordering]

fig_time, ax_time = plt.subplots()

n_steps=4
stepwise_linefits = {step_num: {'prefactor': None, 'exponent': None} for step_num in range(n_steps)}
stepwise_linefits_mem = {step_num: {'prefactor': None, 'exponent': None} for step_num in range(n_steps)}
stepwise_linefits_mem_nonzero = {step_num: {'prefactor': None, 'exponent': None} for step_num in range(n_steps)}
summed_yfit = None
summed_yfit_mem = None
for ind_step in range(n_steps):
    n_secs_total = scaling_df['total_time_per_step_{}'.format(ind_step)].values
    n_secs_total = n_secs_total[ordering]
    n_genes = scaling_df['n_genes'].values
    n_genes = n_genes[ordering]
    n_secs_total_normalized = n_secs_total / n_genes * 2000
    
    # Make a plot for time-scaling
    skip_first_n = 0
    subtract_init = 1000 if ind_step == 2 else 0
        
    x_fit, y_fit, slope, intercept = make_lin_fit(n_cells[skip_first_n:] - subtract_init, n_secs_total_normalized[skip_first_n:], log=True)
    stepwise_linefits[ind_step]['prefactor'] = np.exp(intercept)
    stepwise_linefits[ind_step]['exponent'] = slope
    ax_time.plot(n_cells - subtract_init, n_secs_total_normalized, 'o', c=colors[ind_step], lw=2, markersize=10, label="{}".format(steplabel_map[ind_step]) + r': $T = {%.4f} C^{%.2f}$'  % (np.exp(intercept), slope))
    ax_time.plot(np.exp(x_fit), np.exp(y_fit), '--', lw=2, c=colors[ind_step])
    if summed_yfit is None:
        summed_yfit = np.exp(y_fit)
    else:
        summed_yfit += np.exp(y_fit)

# ax_time.plot(np.exp(x_fit), summed_yfit, '-.', lw=4, c='black', label='Sum of steps')

x_fit_linear = np.linspace(np.exp(x_fit.min()), 1e6, 100)
y_multi_step_fit = np.zeros_like(x_fit_linear)
for ind_step, fit_dict in stepwise_linefits.items():
    y_multi_step_fit += fit_dict['prefactor'] * x_fit_linear ** fit_dict['exponent']
ax_time.plot(x_fit_linear, y_multi_step_fit, '--', lw=4, c='gray', alpha=0.5, label='Sum of steps')

ax_time.set_xscale('log')
ax_time.set_yscale('log')
ax_time.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))
ax_time.grid(True)
ax_time.grid(True, which='minor', lw=0.25, linestyle='--')
ax_time.set_xlabel("Number of cells")
ax_time.set_ylabel("Compute time (seconds) \n normalized per 2000 genes.")
ax_time.set_title("Computation time scaling")
plt.savefig(os.path.join(figure_folder, 'core_compute_scaling_stepwise.png'), dpi=300)
plt.savefig(os.path.join(figure_folder, 'core_compute_scaling_stepwise.svg'))

# Make such a plot for memory scaling as well
skip_first_n = 0
fig_mem, ax_mem = plt.subplots()

for ind_step in range(n_steps):
    max_memories_zero = scaling_df['peak_mem_zero_step_{}'.format(ind_step)] * (1024 ** 2) / 1e9
    max_memories_zero = max_memories_zero[ordering]
    max_memories_nonzero = scaling_df['peak_mem_avg_nonzero_step_{}'.format(ind_step)] * (1024 ** 2) / 1e9
    max_memories_nonzero = max_memories_nonzero[ordering]
    x_fit, y_fit, slope, intercept = make_lin_fit(n_cells[skip_first_n:], max_memories_zero[skip_first_n:], log=True)
    stepwise_linefits_mem[ind_step]['prefactor'] = np.exp(intercept)
    stepwise_linefits_mem[ind_step]['exponent'] = slope
    ax_mem.plot(n_cells, max_memories_zero, 'o', c=colors[ind_step], lw=2, markersize=10, label="Main thread ({})".format(steplabel_map[ind_step]) + r': $M = {%.4f} C^{%.2f}$' % (np.exp(intercept), slope))
    ax_mem.plot(np.exp(x_fit), np.exp(y_fit), '--', lw=2, c=colors[ind_step])
    if summed_yfit_mem is None:
        summed_yfit_mem = np.exp(y_fit)
    else:
        summed_yfit_mem += np.exp(y_fit)

    if np.any(np.isnan(max_memories_nonzero[skip_first_n:])):
        continue
    x_fit, y_fit, slope, intercept = make_lin_fit(n_cells[skip_first_n:], max_memories_nonzero[skip_first_n:], log=True)
    stepwise_linefits_mem_nonzero[ind_step]['prefactor'] = np.exp(intercept)
    stepwise_linefits_mem_nonzero[ind_step]['exponent'] = slope
    ax_mem.plot(n_cells, max_memories_nonzero, '*', c=colors[ind_step], lw=2, markersize=10, label="Worker threads ({})".format(steplabel_map[ind_step])  + r': $M = {%.4f} C^{%.2f}$' % (np.exp(intercept), slope))
    ax_mem.plot(np.exp(x_fit), np.exp(y_fit), linestyle='dotted', lw=2, c=colors[ind_step])

x_fit_linear = np.linspace(np.exp(x_fit.min()), 1e6, 100)
y_multi_step_fit_mem = np.zeros_like(x_fit_linear)
max_exponent = max([line_dict['exponent'] for line_dict in stepwise_linefits_mem.values()])
for ind_step, fit_dict in stepwise_linefits_mem.items():
    # y_multi_step_fit_mem += fit_dict['prefactor'] * x_fit_linear ** fit_dict['exponent']
    y_multi_step_fit_mem = np.maximum(y_multi_step_fit_mem, fit_dict['prefactor'] * x_fit_linear ** fit_dict['exponent'])
ax_mem.plot(x_fit_linear, y_multi_step_fit_mem, '--', lw=4, c='gray', alpha=0.5, label='Multi-step fit (Main threads)')

y_multi_step_fit_mem_nonzero = np.zeros_like(x_fit_linear)
for ind_step, fit_dict in stepwise_linefits_mem_nonzero.items():
    if fit_dict['prefactor'] is None:
        continue
    y_multi_step_fit_mem_nonzero = np.maximum(y_multi_step_fit_mem_nonzero, fit_dict['prefactor'] * x_fit_linear ** fit_dict['exponent'])
ax_mem.plot(x_fit_linear, y_multi_step_fit_mem_nonzero, linestyle='dotted', lw=4, c='gray', label='Multi-step fit (Worker threads)', alpha=.5)

ax_mem.set_xscale('log')
ax_mem.set_yscale('log')
ax_mem.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))
ax_mem.grid(True)
ax_mem.grid(True, which='minor', lw=0.25, linestyle='--')
ax_mem.set_xlabel("Number of cells")
ax_mem.set_ylabel("Memory usage (GB).")
ax_mem.set_title("Memory scaling")
# ax_mem.set_title(r'Fitted scaling: $T = {%.4f} |C|^{%.2f}$' % (np.exp(intercept), slope))
plt.savefig(os.path.join(figure_folder, 'memory_scaling_stepwise.png'), dpi=300)
plt.savefig(os.path.join(figure_folder, 'memory_scaling_stepwise.svg'))

## Now make a plot in which we combine the normal Bonsai scaling and the iterative

In [ ]:
# Pick built-in colormaps
main_colors = [plt.get_cmap("Set1")(i) for i in range(2)]
step_colors = [plt.get_cmap("tab10")(i) for i in range(4)]
for ind in range(len(step_colors)):
    color = list(step_colors[ind])
    color[3] = 0.6
    step_colors[ind] = color

In [ ]:
fig_time, ax_time = plt.subplots()
fig_mem, ax_mem = plt.subplots()

In [ ]:
n_cells_all = []

In [ ]:
# Plot data for normal Bonsai
scaling_df = pd.read_csv('timescaling_normal.csv')
figure_folder = 'figures'

# fig_total, ax_total = plt.subplots()

# Plot timescaling n_cells vs time, but normalized to n_genes
n_cells = scaling_df['n_cells'].values
n_cells_all.append(n_cells)
ordering = np.flatnonzero(~np.isnan(n_cells))[np.argsort(n_cells[~np.isnan(n_cells)])]
n_cells = n_cells[ordering]

n_secs_total = scaling_df['total_time'].values
n_secs_total = n_secs_total[ordering]
n_genes = scaling_df['n_genes'].values
n_genes = n_genes[ordering]
n_secs_total_normalized = n_secs_total / n_genes * 2000
max_memories_zero = scaling_df['peak_mem_zero'] * (1024 ** 2) / 1e9
max_memories_zero = max_memories_zero[ordering]
max_memories_nonzero = scaling_df['peak_mem_avg_nonzero'] * (1024 ** 2) / 1e9
max_memories_nonzero = max_memories_nonzero[ordering]

skip_first_n = 2
x_fit, y_fit, slope, intercept = make_lin_fit(n_cells[skip_first_n:], n_secs_total_normalized[skip_first_n:], log=True)
ax_time.plot(n_cells, n_secs_total_normalized, 'o', c=main_colors[0], lw=2, markersize=10, label="Default Bonsai")
ax_time.plot(np.exp(x_fit), np.exp(y_fit), '--', lw=2, c=main_colors[0], label=r'$T = {%.4f} |C|^{%.2f}$'  % (np.exp(intercept), slope))

# ax_time.set_xscale('log')
# ax_time.set_yscale('log')
# ax_time.legend()
# ax_time.grid(True)
# ax_time.grid(True, which='minor', lw=0.25, linestyle='--')
# ax_time.set_xlabel("Number of cells")
# ax_time.set_ylabel("Compute time (seconds) \n normalized per 2000 genes.")
# ax_time.set_title(r'Fitted scaling: $T = {%.4f} |C|^{%.2f}$' % (np.exp(intercept), slope))
# plt.savefig(os.path.join(figure_folder, 'core_compute_scaling.png'), dpi=300)
# plt.savefig(os.path.join(figure_folder, 'core_compute_scaling.svg'))

# Make such a plot for memory scaling as well
skip_first_n = 2
x_fit, y_fit, slope, intercept = make_lin_fit(n_cells[skip_first_n:], max_memories_zero[skip_first_n:], log=True)
ax_mem.plot(n_cells, max_memories_zero, 'o', c=main_colors[0], lw=2, markersize=10, label=r'Main thread (Default Bonsai)')
ax_mem.plot(np.exp(x_fit), np.exp(y_fit), '--', lw=2, c=main_colors[0], label=r'$M = {%.4f} |C|^{%.2f}$' % (np.exp(intercept), slope))

x_fit, y_fit, slope, intercept = make_lin_fit(n_cells[skip_first_n:], max_memories_nonzero[skip_first_n:], log=True)
ax_mem.plot(n_cells, max_memories_nonzero, '*', c=main_colors[0], lw=2, markersize=10, label=r'Worker threads (Default Bonsai)')
ax_mem.plot(np.exp(x_fit), np.exp(y_fit), '-.', lw=2, c=main_colors[0], label=r'$M = {%.4f} |C|^{%.2f}$' % (np.exp(intercept), slope))

# Add in 2 times the data-file size, which is the memory just needed to load the means and error-bars
datafile_size = scaling_df['datafile_size']
datafile_size = datafile_size[ordering]
ax_mem.plot(n_cells, datafile_size * 2, '-*', lw=1, c='black')
print(max_memories_zero / datafile_size)

# ax_mem.set_xscale('log')
# ax_mem.set_yscale('log')
# ax_mem.legend()
# ax_mem.grid(True)
# ax_mem.grid(True, which='minor', lw=0.25, linestyle='--')
# ax_mem.set_xlabel("Number of cells")
# ax_mem.set_ylabel("Memory usage (GB).")
# # ax_mem.set_title(r'Fitted scaling: $T = {%.4f} |C|^{%.2f}$' % (np.exp(intercept), slope))
# plt.savefig(os.path.join(figure_folder, 'memory_scaling.png'), dpi=300)
# plt.savefig(os.path.join(figure_folder, 'memory_scaling.svg'))


In [ ]:
# Plot data for iterative Bonsai
scaling_df = pd.read_csv('timescaling_iterative.csv')
figure_folder = 'figures'

# fig_total, ax_total = plt.subplots()

# Plot timescaling n_cells vs time, but normalized to n_genes
n_cells = scaling_df['n_cells'].values
n_cells_all.append(n_cells)
ordering = np.flatnonzero(~np.isnan(n_cells))[np.argsort(n_cells[~np.isnan(n_cells)])]
n_cells = n_cells[ordering]
# date_format = "%B %d, %H:%M:%S"
# n_secs_total = []
# for row_ind in range(scaling_df.shape[0]):
#     final_datetime = datetime.strptime(scaling_df['end_time_metadata'].iloc[row_ind], date_format)
#     first_datetime = datetime.strptime(scaling_df['start_time_preprocess'].iloc[row_ind], date_format)
#     n_secs_total.append((final_datetime - first_datetime).total_seconds())
n_secs_total = scaling_df['total_time'].values
n_secs_total = n_secs_total[ordering]
n_genes = scaling_df['n_genes'].values
n_genes = n_genes[ordering]
n_secs_total_normalized = n_secs_total / n_genes * 2000
max_memories_zero = scaling_df['peak_mem_zero'] * (1024 ** 2) / 1e9
max_memories_zero = max_memories_zero[ordering]
max_memories_nonzero = scaling_df['peak_mem_avg_nonzero'] * (1024 ** 2) / 1e9
max_memories_nonzero = max_memories_nonzero[ordering]
# Make a plot for time-scaling
skip_first_n = 0

x_fit, y_fit, slope, intercept = make_lin_fit(n_cells[skip_first_n:], n_secs_total_normalized[skip_first_n:], log=True)
x_fit_linear = np.linspace(np.exp(x_fit.min()), 5e6, 100)
y_multi_step_fit = np.zeros_like(x_fit_linear)
for ind_step, fit_dict in stepwise_linefits.items():
    if ind_step == 2:
        y_multi_step_fit += fit_dict['prefactor'] * (x_fit_linear-1000) ** fit_dict['exponent']
    else:
        y_multi_step_fit += fit_dict['prefactor'] * x_fit_linear ** fit_dict['exponent']
max_exponent = max([line_dict['exponent'] for line_dict in stepwise_linefits.values()])
# ax_time.plot(n_cells, n_secs_total_normalized, 'o', c=main_colors[1], lw=2, markersize=10, label="Iterative Bonsai" + r': $T = {%.4f} C^{%.2f}$'  % (np.exp(intercept), slope))
ax_time.plot(n_cells, n_secs_total_normalized, 'o', c=main_colors[1], lw=2, markersize=10, label="Backbone-based Bonsai")
# ax_time.plot(np.exp(x_fit), np.exp(y_fit), '--', lw=2, c=main_colors[1])
ax_time.plot(x_fit_linear, y_multi_step_fit, '--', lw=2, c=main_colors[1], label=r'Multi-step fit: $T \rightarrow C^{%.2f}$' % max_exponent)

# Make such a plot for memory scaling as well
skip_first_n = 0
# fig_mem, ax_mem = plt.subplots()
x_fit, y_fit, slope, intercept = make_lin_fit(n_cells[skip_first_n:], max_memories_zero[skip_first_n:], log=True)
x_fit_linear = np.linspace(np.exp(x_fit.min()), 5e6, 100)
y_multi_step_fit = np.zeros_like(x_fit_linear)
for ind_step, fit_dict in stepwise_linefits_mem.items():
    y_multi_step_fit = np.maximum(y_multi_step_fit, fit_dict['prefactor'] * x_fit_linear ** fit_dict['exponent'])
max_exponent = max([line_dict['exponent'] for line_dict in stepwise_linefits_mem.values()])
ax_mem.plot(n_cells, max_memories_zero, 'o', c=main_colors[1], lw=2, markersize=10, label=r'Main thread (Backbone-based Bonsai):')
# ax_mem.plot(np.exp(x_fit), np.exp(y_fit), '--', lw=2, c=main_colors[1])
ax_mem.plot(x_fit_linear, y_multi_step_fit, '--', lw=2, c=main_colors[1], label=r'Multi-step fit: $M \rightarrow C^{%.2f}$' % max_exponent)


x_fit, y_fit, slope, intercept = make_lin_fit(n_cells[skip_first_n:], max_memories_nonzero[skip_first_n:], log=True)
x_fit_linear = np.linspace(np.exp(x_fit.min()), 5e6, 100)
y_multi_step_fit = np.zeros_like(x_fit_linear)
for ind_step, fit_dict in stepwise_linefits_mem_nonzero.items():
    if fit_dict['prefactor'] is not None:
        y_multi_step_fit = np.maximum(y_multi_step_fit, fit_dict['prefactor'] * x_fit_linear ** fit_dict['exponent'])
max_exponent = max([line_dict['exponent'] for line_dict in stepwise_linefits_mem_nonzero.values() if line_dict['prefactor']])
ax_mem.plot(n_cells, max_memories_nonzero, '*', c=main_colors[1], lw=2, markersize=10, label=r'Worker threads (Backbone-based Bonsai)')
# ax_mem.plot(np.exp(x_fit), np.exp(y_fit), '-.', lw=2, c=main_colors[1])
ax_mem.plot(x_fit_linear, y_multi_step_fit, '-.', lw=2, c=main_colors[1], label=r'Multi-step fit: $M \rightarrow C^{%.2f}$' % max_exponent)

datafile_size = scaling_df['datafile_size']
datafile_size = datafile_size[ordering]
ax_mem.plot(n_cells, datafile_size * 2, '-*', lw=1, c='black', label='Size of input data')
print(max_memories_zero / datafile_size)

# # Add in 2 times the data-file size, which is the memory just needed to load the means and error-bars
# datafile_size = scaling_df['datafile_size']
# datafile_size = datafile_size[ordering]
# ax_mem.plot(n_cells, datafile_size * 2, '-*', lw=1, c='black')

# ax_mem.set_title(r'Fitted scaling: $T = {%.4f} |C|^{%.2f}$' % (np.exp(intercept), slope))


In [ ]:
n_cells = np.unique(np.hstack(n_cells_all))
n_cells = n_cells[~np.isnan(n_cells)].astype(dtype=int)

In [ ]:
stepwise_linefits

In [ ]:
def cell_tick_formatter(x, pos):
    # Powers of two → 2^n
    if x > 0 and np.log2(x).is_integer():
        return r"$2^{{{}}}$".format(int(np.log2(x)))

    # Special large values
    if x == 500_000:
        return "500K"
    if x == 1_000_000:
        return "1M"
    if x == 5_000_000:
        return "5M"

    return ""


def time_tick_formatter(y, pos):
    if y == 60:
        return "1 min"
    if y == 3600:
        return "1 hour"
    if y == 86400:
        return "1 day"
    if y == 600:
        return "10 mins"
    if y == 36000:
        return "10 hours"
    if y == 7 * 86400:
        return "1 week"
    return ""


def mem_tick_formatter(y, pos):
    if y == 60:
        return "1 min"
    if y == 3600:
        return "1 hour"
    if y == 86400:
        return "1 day"
    if y == 600:
        return "10 mins"
    if y == 36000:
        return "10 hours"
    if y == 7 * 86400:
        return "1 week"
    return ""

In [ ]:
ax_time.set_xscale('log')
ax_time.set_yscale('log')
ax_time.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))
ax_time.grid(True)
ax_time.grid(True, which='minor', lw=0.25, linestyle='--')
ax_time.set_xlabel("Number of cells")
ax_time.set_ylabel("Compute time (seconds) \n normalized per 2000 genes.")

extra_ticks = np.array([500000, 1000000, 5000000])
tick_positions = np.hstack((n_cells, extra_ticks))
ax_time.set_xticks(tick_positions)
ax_time.xaxis.set_minor_locator(mticker.NullLocator())
ax_time.xaxis.set_major_formatter(mticker.FuncFormatter(cell_tick_formatter))
# ax_time.xaxis.set_major_formatter(mticker.ScalarFormatter())
# ax_time.ticklabel_format(axis="x", style="plain")
# plt.setp(ax_time.get_xticklabels(), rotation=45, ha="center", rotation_mode="anchor")
plt.setp(ax_time.get_xticklabels(), rotation=45, ha="right")
ax_time.tick_params(axis="x", pad=6)
ax_time.set_ylim(top=6e5)

y_ticks = [
    60,                # 1 minute
    600,
    3600,              # 1 hour
    36000,
    86400,             # 1 day
    7 * 86400          # 1 week
]

ax_time.set_yticks(y_ticks)
ax_time.yaxis.set_major_formatter(
    mticker.FuncFormatter(time_tick_formatter)
)
ax_time.yaxis.set_minor_locator(mticker.NullLocator())

# ax_time.set_title(r'Fitted scaling: $T = {%.4f} |C|^{%.2f}$' % (np.exp(intercept), slope))
fig_time = ax_time.get_figure()  # get the Figure containing ax_time
fig_time.savefig(os.path.join(figure_folder, 'comparison_standard_and_iterative_computetime_scaling.png'), dpi=300, bbox_inches='tight')
fig_time.savefig(os.path.join(figure_folder, 'comparison_standard_and_iterative_computetime_scaling.svg'), bbox_inches='tight')

ax_mem.set_xscale('log')
ax_mem.set_yscale('log')
ax_mem.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))
ax_mem.grid(True)
ax_mem.grid(True, which='minor', lw=0.25, linestyle='--')
ax_mem.set_xlabel("Number of cells")
ax_mem.set_ylabel("Memory usage (GB).")

ax_mem.set_xticks(n_cells)
ax_mem.xaxis.set_minor_locator(mticker.NullLocator())
ax_mem.xaxis.set_major_formatter(mticker.ScalarFormatter())
ax_mem.ticklabel_format(axis="x", style="plain")
plt.setp(ax_mem.get_xticklabels(), rotation=45, ha="right")

ax_mem.set_xticks(tick_positions)
ax_mem.xaxis.set_minor_locator(mticker.NullLocator())
ax_mem.xaxis.set_major_formatter(mticker.FuncFormatter(cell_tick_formatter))
# ax_time.xaxis.set_major_formatter(mticker.ScalarFormatter())
# ax_time.ticklabel_format(axis="x", style="plain")
# plt.setp(ax_time.get_xticklabels(), rotation=45, ha="center", rotation_mode="anchor")
plt.setp(ax_time.get_xticklabels(), rotation=45, ha="right")
ax_mem.tick_params(axis="x", pad=6)

fig_mem = ax_mem.get_figure()  # get the Figure containing ax_mem
fig_mem.savefig(os.path.join(figure_folder, 'comparison_standard_and_iterative_memory_scaling.png'), dpi=300, bbox_inches='tight')
fig_mem.savefig(os.path.join(figure_folder, 'comparison_standard_and_iterative_memory_scaling.svg'), bbox_inches='tight')

In [ ]:
fig_time

In [ ]:
fig_mem

## Now compare the loglikelihoods across datasets

In [ ]:
scaling_df_normal = pd.read_csv('timescaling_normal.csv')
scaling_df_iterative = pd.read_csv('timescaling_iterative.csv')


In [ ]:
# common_datasets = scaling_df_normal["dataset"].intersection(scaling_df_iterative["dataset"])
common_datasets = set(scaling_df_normal["dataset"]) & set(scaling_df_iterative["dataset"])
common_datasets = [dataset for dataset in common_datasets if ~np.isnan(scaling_df_normal[scaling_df_normal['dataset'] == dataset]['loglikelihood_final'].values)]
cols = ["dataset", "n_cells", "n_genes", "loglikelihood_final"]
df_normal_common = scaling_df_normal.loc[
    scaling_df_normal["dataset"].isin(common_datasets), cols].copy()

df_iterative_common = scaling_df_iterative.loc[
    scaling_df_iterative["dataset"].isin(common_datasets), cols].copy()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Normalize log-likelihood
df_normal_common["ll_norm"] = (
    df_normal_common["loglikelihood_final"]
    / (df_normal_common["n_cells"] * df_normal_common["n_genes"])
)

df_iterative_common["ll_norm"] = (
    df_iterative_common["loglikelihood_final"]
    / (df_iterative_common["n_cells"] * df_iterative_common["n_genes"])
)

# Sort consistently (choose what you want the order to mean)
df_normal_common = df_normal_common.sort_values("n_cells")
df_iterative_common = df_iterative_common.sort_values("n_cells")

# Categorical x positions
x = np.arange(len(df_normal_common))

fig, ax = plt.subplots(figsize=(8, 4))

ax.plot(
    x,
    df_normal_common["ll_norm"].values,
    marker="o",
    label="Normal",
    color=main_colors[0],
)

ax.plot(
    x,
    df_iterative_common["ll_norm"].values,
    marker="o",
    label="Iterative",
    color=main_colors[1],
)

# Set categorical ticks with n_cells labels
ax.set_xticks(x)
ax.set_xticklabels(df_normal_common["n_cells"].astype(int).astype(str))

ax.set_xlabel("Number of cells")
ax.set_ylabel("Log-likelihood / (n_cells × n_genes)")

ax.legend()
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

plt.tight_layout()
plt.show()


In [ ]:
df_iterative_common

In [ ]:
df_normal_common

## Finally, compare pairwise distance reconstruction